In [0]:
from pyspark.sql.functions import (
    col,
    lag,
    round,
    current_timestamp,
    when,
    count
)
from pyspark.sql.window import Window

df_silver_pop = spark.read.table("silver.worldbank_population")
df_silver_countries = spark.read.table("silver.worldbank_countries")

duplicate_metadata = (
    df_silver_countries
    .groupBy("country_code")
    .count()
    .filter(col("count") > 1)
)

assert duplicate_metadata.count() == 0, (
    "Duplicate country metadata found before Gold join"
)

unmatched_population = (
    df_silver_pop
    .select("country_code")
    .distinct()
    .join(
        df_silver_countries.select("country_code").distinct(),
        on="country_code",
        how="left_anti"
    )
)

print(
    "Population codes without matching economy metadata:",
    unmatched_population.count()
)

df_gold_joined = (
    df_silver_pop.alias("pop")
    .join(
        df_silver_countries.alias("countries"),
        col("pop.country_iso3") == col("countries.country_code"),
        "inner"
    )
    .select(
        col("countries.country_name"),
        col("countries.region_name").alias("current_region"),
        col("countries.income_level").alias("current_income_group"),
        col("pop.year"),
        col("pop.population"),
        col("countries.country_code"),
        col("countries.latitude"),
        col("countries.longitude"),
        col("countries.raw_ingested_at").alias("metadata_retrieved_at")
    )
)

duplicate_gold_records = (
    df_gold_joined
    .groupBy("country_code", "year")
    .count()
    .filter(col("count") > 1)
)

assert duplicate_gold_records.count() == 0, (
    "Duplicate economy-year records found after Gold join"
)

window_spec = (
    Window
    .partitionBy("country_code")
    .orderBy("year")
)

df_gold_analytics = (
    df_gold_joined
    .withColumn(
        "prev_year_population",
        lag("population", 1).over(window_spec)
    )
    .withColumn(
        "prev_year",
        lag("year", 1).over(window_spec)
    )
    .withColumn(
        "yoy_growth_percentage",
        when(
            (col("year") - col("prev_year") == 1) &
            (col("prev_year_population") > 0),
            round(
                (
                    (col("population") - col("prev_year_population")) /
                    col("prev_year_population")
                ) * 100,
                2
            )
        )
    )
    .drop("prev_year")
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

assert (
    df_gold_analytics.filter(
        col("country_code").isNull() |
        col("year").isNull() |
        col("population").isNull()
    ).count() == 0
), "Null key or population values found in Gold"

assert (
    df_gold_analytics.filter(
        col("country_code") == "1W"
    ).count() == 0
), "World aggregate must not be present in Gold"

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

(
    df_gold_analytics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.population_analytics")
)

display(spark.sql("""
    SELECT *
    FROM gold.population_analytics
    WHERE country_name = 'Brazil'
    ORDER BY year DESC
    LIMIT 10
"""))